# Vehicle Assembly Anomaly Benchmark

Compares LightGBM, LSTM, and Transformer. Each dataset uses vehicle-level splitting, cross-validation for hyperparameter selection on training data, then one final held-out test evaluation.
    "metadata": {"id": "07a43b37", "language": "markdown"},

In [1]:
import sys, subprocess, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
try:
    import lightgbm as lgb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm'])
    import lightgbm as lgb
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Notebook Python:', sys.executable)
print('LightGBM:', lgb.__version__, '| PyTorch:', torch.__version__, '| Device:', DEVICE)

Notebook Python: c:\Users\prath\AppData\Local\Programs\Python\Python311\python.exe
LightGBM: 4.7.0 | PyTorch: 2.7.1+cu118 | Device: cuda


In [2]:
DATA_DIR = Path.cwd()
DATASETS = {name: DATA_DIR / filename for name, filename in {'base': 'dataset.csv', 'sparse': 'dataset_variant_sparse.csv', 'drift': 'dataset_variant_drift.csv', 'propagation': 'dataset_variant_propagation.csv', 'noisy': 'dataset_variant_noisy.csv'}.items()}
TARGETS = ['anomaly_flag', 'root_cause_station']
VEHICLE_ID = 'vehicle_id'
TEST_SIZE = 0.2
N_SPLITS = 3
NUMERIC_COLS = ['cycle_time_sec', 'torque_nm', 'temperature_c', 'vibration_rms', 'pressure_bar', 'force_n', 'position_error_mm', 'voltage_v', 'current_a', 'flow_rate_lpm', 'queue_time_sec', 'ambient_temperature_c', 'humidity_pct']
CATEGORICAL_COLS = ['vehicle_model', 'vehicle_variant', 'station_id', 'shift', 'production_batch']
FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS
for name, path in DATASETS.items():
    print(name, path.exists(), path)

base True c:\Users\prath\Documents\Accenture\dataset.csv
sparse True c:\Users\prath\Documents\Accenture\dataset_variant_sparse.csv
drift True c:\Users\prath\Documents\Accenture\dataset_variant_drift.csv
propagation True c:\Users\prath\Documents\Accenture\dataset_variant_propagation.csv
noisy True c:\Users\prath\Documents\Accenture\dataset_variant_noisy.csv


In [6]:
def fit_torch(X, y, X_eval, params, kind, num_classes=1):
    model = SequenceModel(X.shape[2], params['hidden'], params['layers'], params['dropout'], kind, num_classes).to(DEVICE)
    if num_classes == 1:
        pos, neg = max(float(y.sum()), 1), max(float(len(y) - y.sum()), 1)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg / pos], device=DEVICE))
    else:
        loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=params['lr'])
    loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)), batch_size=params['batch'], shuffle=True)
    for _ in range(params['epochs']):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb.to(DEVICE))
            if num_classes == 1:
                loss = loss_fn(logits.squeeze(-1), yb.float().to(DEVICE))
            else:
                loss = loss_fn(logits, yb.to(DEVICE))
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_eval).to(DEVICE))
        if num_classes == 1:
            return torch.sigmoid(logits.squeeze(-1)).cpu().numpy()
        else:
            return torch.softmax(logits, dim=1).cpu().numpy()

In [7]:
def benchmark(name, df, target):
    train_df, test_df = split_vehicles(df, target)
    train_matrix, test_matrix = preprocess(train_df, test_df)
    X, y, encoder = sequences(train_df, train_matrix, target)
    Xt, yt, _ = sequences(test_df, test_matrix, target, encoder)
    is_multiclass = target == 'root_cause_station'
    num_classes = len(encoder.classes_) if is_multiclass else 1
    grids = GRIDS_MULTICLASS if is_multiclass else GRIDS_BINARY
    cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED) if not is_multiclass else StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    output = []
    for model_name in MODEL_NAMES:
        best_score, best = -np.inf, None
        score_metric = 'macro_f1' if is_multiclass else 'roc_auc'
        for params in ParameterGrid(grids[model_name]):
            fold_scores = []
            for a, b in cv.split(X, y):
                if model_name == 'lightgbm':
                    obj = 'multiclass' if is_multiclass else 'binary'
                    params_copy = params.copy()
                    if is_multiclass:
                        params_copy['num_class'] = num_classes
                    model = lgb.LGBMClassifier(objective=obj, class_weight='balanced' if not is_multiclass else None, verbosity=-1, random_state=SEED, **params_copy)
                    model.fit(X[a].reshape(len(a), -1), y[a])
                    if is_multiclass:
                        p = model.predict_proba(X[b].reshape(len(b), -1))
                    else:
                        p = model.predict_proba(X[b].reshape(len(b), -1))[:, 1]
                else:
                    p = fit_torch(X[a], y[a], X[b], params, model_name, num_classes)
                if is_multiclass:
                    fold_scores.append(f1_score(y[b], np.argmax(p, axis=1), average='macro', zero_division=0))
                else:
                    fold_scores.append(roc_auc_score(y[b], p))
            if np.mean(fold_scores) > best_score:
                best_score, best = float(np.mean(fold_scores)), params
        if model_name == 'lightgbm':
            obj = 'multiclass' if is_multiclass else 'binary'
            params_copy = best.copy()
            if is_multiclass:
                params_copy['num_class'] = num_classes
            final = lgb.LGBMClassifier(objective=obj, class_weight='balanced' if not is_multiclass else None, verbosity=-1, random_state=SEED, **params_copy)
            final.fit(X.reshape(len(X), -1), y)
            if is_multiclass:
                p = final.predict_proba(Xt.reshape(len(Xt), -1))
            else:
                p = final.predict_proba(Xt.reshape(len(Xt), -1))[:, 1]
        else:
            p = fit_torch(X, y, Xt, best, model_name, num_classes)
        result = {'dataset': name, 'target': target, 'model': model_name, f'cv_{score_metric}': best_score, 'best_params': str(best), **scores(yt, p, is_multiclass)}
        output.append(result)
        print(f'{name:12} | {target:20} | {model_name:12} | CV {score_metric}={best_score:.4f} | Test Acc={result["accuracy"]:.4f}')
    return output

all_results = []
for target in TARGETS:
    print(f'\n===== BENCHMARKING {target.upper()} =====')
    for name, path in DATASETS.items():
        all_results.extend(benchmark(name, load_dataset(path), target))
all_results_df = pd.DataFrame(all_results)


===== BENCHMARKING ANOMALY_FLAG =====
base         | anomaly_flag         | lightgbm     | CV roc_auc=1.0000 | Test Acc=1.0000
base         | anomaly_flag         | lstm         | CV roc_auc=1.0000 | Test Acc=0.9995
base         | anomaly_flag         | transformer  | CV roc_auc=1.0000 | Test Acc=1.0000
sparse       | anomaly_flag         | lightgbm     | CV roc_auc=0.9876 | Test Acc=0.9925
sparse       | anomaly_flag         | lstm         | CV roc_auc=0.9903 | Test Acc=0.9910
sparse       | anomaly_flag         | transformer  | CV roc_auc=0.9894 | Test Acc=0.9970
drift        | anomaly_flag         | lightgbm     | CV roc_auc=0.9999 | Test Acc=1.0000
drift        | anomaly_flag         | lstm         | CV roc_auc=1.0000 | Test Acc=1.0000
drift        | anomaly_flag         | transformer  | CV roc_auc=1.0000 | Test Acc=1.0000
propagation  | anomaly_flag         | lightgbm     | CV roc_auc=1.0000 | Test Acc=0.9985
propagation  | anomaly_flag         | lstm         | CV roc_auc=1.0000 

In [8]:
print('\n===== ANOMALY DETECTION RESULTS =====')
anomaly_df = all_results_df[all_results_df['target'] == 'anomaly_flag'].copy()
display(anomaly_df[['dataset', 'model', 'roc_auc', 'f1', 'recall']].sort_values(['dataset', 'roc_auc'], ascending=[True, False]))
anomaly_summary = anomaly_df.groupby('model')[['roc_auc', 'f1', 'recall']].mean().sort_values('roc_auc', ascending=False)
print('\nBest models for ANOMALY DETECTION (by mean ROC-AUC):')
display(anomaly_summary)

print('\n===== ROOT CAUSE DETECTION RESULTS =====')
rootcause_df = all_results_df[all_results_df['target'] == 'root_cause_station'].copy()
display(rootcause_df[['dataset', 'model', 'accuracy', 'macro_f1', 'weighted_f1']].sort_values(['dataset', 'accuracy'], ascending=[True, False]))
rootcause_summary = rootcause_df.groupby('model')[['accuracy', 'macro_f1', 'weighted_f1']].mean().sort_values('macro_f1', ascending=False)
print('\nBest models for ROOT CAUSE DETECTION (by mean Macro F1):')
display(rootcause_summary)

print('\n===== OVERALL COMPARISON =====')
comparison = pd.DataFrame({
    'Anomaly Detection (ROC-AUC)': anomaly_summary['roc_auc'],
    'Root Cause Detection (Macro F1)': rootcause_summary['macro_f1']
})
display(comparison)

all_results_df.to_csv(DATA_DIR / 'model_comparison_results_full.csv', index=False)
print('\nResults saved to model_comparison_results_full.csv')


===== ANOMALY DETECTION RESULTS =====


,dataset,model,roc_auc,f1,recall
0,base,lightgbm,1.000000,1.000000,1.000000
2,base,transformer,1.000000,1.000000,1.000000
1,base,lstm,0.999996,0.996656,0.993333
6,drift,lightgbm,1.000000,1.000000,1.000000
7,drift,lstm,1.000000,1.000000,1.000000
8,drift,transformer,1.000000,1.000000,1.000000
12,noisy,lightgbm,0.994620,0.948454,0.920000
14,noisy,transformer,0.992951,0.976271,0.960000
13,noisy,lstm,0.981503,0.947020,0.953333
9,propagation,lightgbm,1.000000,0.989899,0.980000



Best models for ANOMALY DETECTION (by mean ROC-AUC):


,roc_auc,f1,recall
model,,,
transformer,0.998412,0.991281,0.989333
lightgbm,0.997906,0.977144,0.960000
lstm,0.995715,0.976601,0.985333



===== ROOT CAUSE DETECTION RESULTS =====


...


Best models for ROOT CAUSE DETECTION (by mean Macro F1):


,accuracy,macro_f1,weighted_f1
model,,,
lstm,0.9974,0.952566,0.997222
transformer,0.9947,0.931811,0.994359
lightgbm,0.9951,0.906242,0.994411



===== OVERALL COMPARISON =====


,Anomaly Detection (ROC-AUC),Root Cause Detection (Macro F1)
model,,
lightgbm,0.997906,0.906242
lstm,0.995715,0.952566
transformer,0.998412,0.931811



Results saved to model_comparison_results_full.csv
